In [1]:
import time
import numpy as np

In [2]:
import dask
import dask.array

In [3]:
import h5py
import os
from datetime import datetime

In [4]:
from scipy import ndimage

In [5]:
sample = '024_large_02_3uls'
scan = '0004'
toppath = '/data/visitor/ihmi1620/id19/20261006'

# input paths
procpath = os.path.join(toppath, 'PROCESSED_DATA')
samplepath = os.path.join(procpath, sample, sample+'_'+scan)
int16_volumepath = os.path.join(samplepath, 'reconstructed_volumes','cast_volume')

In [6]:
def load_im_dask(datapath, name, ts=None):
    splitname = name.split('_')
    ts = int(splitname[-3][:-3])
    entry = '/entry'+f'{ts:04d}'+'/reconstruction/results/data'
    tsentry =  '/entry'+f'{ts:04d}'+'/reconstruction/configuration/acquisition/end_time'
    # print(name, entry, tsentry)
    path = os.path.join(datapath, name)
    file = h5py.File(path)
    da = dask.array.from_array(file[entry])
    byte_data = file[tsentry][()]
    timestamp = datetime.fromisoformat(byte_data.decode()).timestamp() # ChatGPT
    return da, timestamp

In [7]:
def stack_4D(datapath):
    names = os.listdir(datapath)
    names.sort()
    data = []
    timestamps = []
    for name in names:
        da, timestamp = load_im_dask(datapath, name)
        data.append(da)
        timestamps.append(timestamp)
    timestamps = np.array(timestamps)
    Stack = dask.array.stack(data, axis=3)
    return Stack, names, timestamps

In [8]:
Stack, names, timestamps = stack_4D(int16_volumepath)

In [9]:
im = Stack[:,:,:,30]

In [10]:
im = im.compute()

In [11]:
im.shape

(3008, 1024, 1024)

In [12]:
subim = im[1000:1600,200:600,200:600]

In [25]:
t0 = time.time()
grad = ndimage.gaussian_filter(subim, 1)
grad = np.gradient(grad, axis = 2)
print(time.time()-t0)

1.2445666790008545


In [24]:
t0 = time.time()
grad = ndimage.gaussian_filter(subim, 1, order=(0, 0, 1), mode="nearest")
print(time.time()-t0)

1.0748894214630127


In [11]:
subim

dask.array<getitem, shape=(600, 400, 400), dtype=uint16, chunksize=(382, 206, 206), chunktype=numpy.ndarray>

In [13]:
volume = dask.array.from_array(subim)
# volume = subim

In [14]:
volume

dask.array<array, shape=(600, 400, 400), dtype=uint16, chunksize=(419, 400, 400), chunktype=numpy.ndarray>

In [140]:
import dask.array as lib
import dask_image.ndfilters

In [16]:
sigma = 1
rho = 3
truncate = 4

In [388]:
    Vx = dask_image.ndfilters.gaussian_filter(volume, sigma, order=(0, 0, 1), mode="nearest", truncate=truncate)  # type: ignore
    Vy = dask_image.ndfilters.gaussian_filter(volume, sigma, order=(0, 1, 0), mode="nearest", truncate=truncate)  # type: ignore
    Vz = dask_image.ndfilters.gaussian_filter(volume, sigma, order=(1, 0, 0), mode="nearest", truncate=truncate)  # type: ignore  
    
    S = [] # append to list instead of pre-allocate memory, dask takes care of thos
    
    # Integrating elements of structure tensor (scipy uses sequence of 1D).
    S.append(lib.multiply(Vx, Vx))
    S.append(lib.multiply(Vy, Vy))
    S.append(lib.multiply(Vz, Vz))
    S.append(lib.multiply(Vx, Vy))
    S.append(lib.multiply(Vx, Vz))
    S.append(lib.multiply(Vy, Vz))
    
    for i in range(6):
        S[i] = dask_image.ndfilters.gaussian_filter(S[i], rho, mode="nearest", truncate=truncate)
        
    S = lib.stack(S)

In [110]:
import matplotlib.pyplot as plt

In [183]:
test = S[:,300,200,200].compute()

In [215]:
test

array([30213, 31407, 32177, 32611, 33706, 33313], dtype=uint16)

In [190]:
ST_test = np.array([[test[0],test[3], test[4]], [test[3],test[1], test[5]], [test[4],test[5], test[2]]])

In [231]:
import numpy as lib
eig_special_3d(np.float32(test))

(array([97698.125 , -1356.4141, -2544.7324], dtype=float32),
 array([ 0.76235354, -0.10419258, -0.63871837], dtype=float32))

In [234]:
eig_special_3d_robert(ST_test)

/tmp/ipykernel_2926771/671356942.py:19: RuntimeWarning: invalid value encountered in sqrt
  phi = lib.atan(lib.sqrt(p**3-q**2))/q/3


array([nan, nan, nan])

In [229]:
ST_test

array([[30213, 32611, 33706],
       [32611, 31407, 33313],
       [33706, 33313, 32177]], dtype=uint16)

In [259]:
def eig_special_3d(
    S: libt.ArrayLike,
    full: bool = False,
    eigenvalue_order: Literal["desc", "asc"] = "desc",
) -> tuple[libt.NDArray, libt.NDArray]:
    """Eigensolution for symmetric real 3-by-3 matrices.

    Args:
        S: A floating point array with shape (6, ...) containing structure tensor. Use `float64` to avoid numerical errors. When using lower precision, ensure that the values of S are not very small/large. Pass `cupy.ndarray` to avoid copying S.
        full: A flag indicating that all three eigenvectors should be returned.
        eigenvalue_order: The order of eigenvalues. Either "desc" for descending or "asc" for ascending. If all three eigenvectors are returned, they will be ordered according to the eigenvalues.

    Returns:
        val: An array with shape `(3, ...)` containing eigenvalues.
        vec: An array with shape `(3, ...)` containing the vector corresponding to the smallest eigenvalue if `full` is `False`, otherwise `(3, 3, ...)` containing eigenvectors.

    More:
        An analytic solution of eigenvalue problem for real symmetric matrix,
        using an affine transformation and a trigonometric solution of third
        order polynomial. See https://en.wikipedia.org/wiki/Eigenvalue_algorithm
        which refers to Smith's algorithm https://dl.acm.org/citation.cfm?id=366316.

    Authors: vand@dtu.dk, 2019; niejep@dtu.dk, 2019-2024
    """
    S = lib.asarray(S)

    # Check data type. Must be floating point.
    if not lib.issubdtype(S.dtype, lib.floating):
        raise ValueError("S must be floating point type.")

    # Flatten S.
    input_shape = S.shape
    S = S.reshape(6, -1)

    # Create v vector.
    v = lib.array([[2 * lib.pi / 3], [4 * lib.pi / 3]], dtype=S.dtype)

    # Computing eigenvalues.

    # Allocate vec and val. We will use them for intermediate computations as well.
    if full:
        val = lib.empty((3,) + S.shape[1:], dtype=S.dtype)
        vec = lib.empty((9,) + S.shape[1:], dtype=S.dtype)
        tmp = lib.empty((4,) + S.shape[1:], dtype=S.dtype)
        B03 = val
        B36 = vec[:3]
    else:
        val = lib.empty((3,) + S.shape[1:], dtype=S.dtype)
        vec = lib.empty((3,) + S.shape[1:], dtype=S.dtype)
        tmp = lib.empty((4,) + S.shape[1:], dtype=S.dtype)
        B03 = val
        B36 = vec

    # Views for B.
    B0 = B03[0]
    B1 = B03[1]
    B2 = B03[2]
    B3 = B36[0]
    B4 = B36[1]
    B5 = B36[2]

    # Compute q, mean of diagonal. We need to use q multiple times later.
    # Using lib.mean has precision issues.
    q = lib.add(S[0], S[1], out=tmp[0])
    q += S[2]
    q /= 3

    # Compute S minus q. Insert it directly into B where it'll stay.
    Sq = lib.subtract(S[:3], q, out=B03)

    # Compute s, off-diagonal elements. Store in part of B not yet used.
    s = lib.einsum("ij,ij->j", S[3:], S[3:], out=tmp[1])
    s *= 2

    # Compute p.
    p = lib.einsum("ij,ij->j", Sq, Sq, out=tmp[2])
    del Sq  # Last use of Sq.
    p += s

    p *= 1 / 6
    lib.sqrt(p, out=p)

    # Compute inverse p, while avoiding 0 division.
    # Reuse s allocation and delete s variable.
    p_inv = s
    del s
    p_inv[:] = 0
    lib.divide(1, p, out=p_inv, where=p != 0)

    # Compute B. First part is already filled.
    B03 *= p_inv
    lib.multiply(S[3:], p_inv, out=B36)

    # Compute d, determinant of B.
    d = lib.prod(B03, axis=0, out=tmp[3])

    # Reuse allocation for p_inv and delete variable.
    d_tmp = p_inv
    del p_inv
    # Computation of d.
    lib.multiply(B2, B3, d_tmp)
    d_tmp *= B3
    d -= d_tmp
    lib.multiply(B4, B4, out=d_tmp)
    d_tmp *= B1
    d -= d_tmp
    lib.prod(B36, axis=0, out=d_tmp)
    d_tmp *= 2
    d += d_tmp
    lib.multiply(B5, B5, out=d_tmp)
    d_tmp *= B0
    d -= d_tmp
    d *= 0.5
    # Ensure -1 <= d/2 <= 1.
    lib.clip(d, -1, 1, out=d)

    # Compute phi. Beware that we reuse d variable!
    phi = d
    del d
    phi = lib.arccos(phi, out=phi)
    phi /= 3

    # Compute val, ordered eigenvalues. Resuing B allocation.
    del B03, B36, B0, B1, B2, B3, B4, B5

    lib.add(v, phi[lib.newaxis], out=val[:2])
    val[2] = phi
    lib.cos(val, out=val)
    p *= 2
    val *= p
    val += q

    val = val.reshape(val.shape[:-1] + input_shape[1:])


    return val

In [382]:
def eig_special_3d_robert(S):   
#     #RF: the original code uses quite a few memory allocations incompitble with dask
#     # therfore the eigenvalue extraction is built directly from https://dl.acm.org/doi/epdf/10.1145/355578.366316
#     # testing shows that the built-in numpy (and dask methods) are faster for a single matrix, but full volume much much faster with QIM method
    
    #TODO: maybe only use half of the symmetric matrix as the original code does
    
    shp = S.shape
    S = S.reshape(6,-1)
    S0 = S[0]
    S1 = S[1]
    S2 = S[2]
    S3 = S[3]
    S4 = S[4]
    S5 = S[5]
    
    
    # 3m = trace(A)
    m = (S0+S1+S2)/3
    
    
    # 6p = "sum of squares of elements of (A-mI)
    Sq = lib.subtract(S[:3], m)
    s = lib.einsum("ij,ij->j", S[3:], S[3:])
    s *= 2
    p = lib.einsum("ij,ij->j", Sq, Sq)
    p += s
    p /= 6
    
    #2q = det(A - mI)
    S0q = Sq[0]
    S1q = Sq[1]
    S2q = Sq[2]
    
    det = S0q*S1q*S2q + S3*S5*S4 + S4*S3*S5 - S4*S1q*S4 - S3*S3*S2q - S0q*S5*S5
    q = det/2
    
    #phi
    phi = lib.arctan(lib.sqrt(p**3-q**2)/q)/3
    
    #eigenvalues
    psqrt = lib.sqrt(p)
    cos = lib.cos(phi)
    sin = lib.sin(phi)
    
    v1 = m + 2*psqrt*cos
    v2 = m - lib.sqrt(p)*(cos + lib.sqrt(3)*sin)
    v3 = m - lib.sqrt(p)*(cos - lib.sqrt(3)*sin)
    
    val = lib.array([v1,v2,v3])

    return val
    # valmax = val.max(axis=0)
    # valmin = val.min(axis=0)
    
    # valmax = valmax.reshape(valmax.shape[:-1] + shp[1:])
    # valmin = valmin.reshape(valmin.shape[:-1] + shp[1:])
    
    # return valmax, valmin

In [236]:
lib.sum(AmI**2)

np.float64(6620614062.666667)

In [238]:
    m = lib.trace(ST_test)/3
    
    # 6p = "sum of squares of elements of (A-mI)
    AmI = ST_test - m*lib.eye(3)
    p = lib.sum(AmI**2)/6
    
    #2q = det(A - mI)
    q = lib.linalg.det(AmI)
    
    #phi
    phi = lib.atan(lib.sqrt(p**3-q**2))/q/3
    
    
    #eigenvalues
    psqrt = lib.sqrt(p)
    cos = lib.cos(phi)
    sin = lib.sin(phi)
    
    v1 = m + 2*psqrt*cos
    v2 = m - lib.sqrt(p)*(cos + lib.sqrt(3)*sin)
    v3 = m - lib.sqrt(p)*(cos - lib.sqrt(3)*sin)
    
    val = lib.array([v1,v2,v3])

/tmp/ipykernel_2926771/710463600.py:11: RuntimeWarning: invalid value encountered in sqrt
  phi = lib.atan(lib.sqrt(p**3-q**2))/q/3


In [239]:
p

np.float64(1103435677.1111112)

In [240]:
q

np.float64(73272708006985.84)

In [241]:
m

np.float64(31265.666666666668)

In [242]:
ST_test

array([[30213, 32611, 33706],
       [32611, 31407, 33313],
       [33706, 33313, 32177]], dtype=uint16)

In [243]:
AmI

array([[-1052.66666667, 32611.        , 33706.        ],
       [32611.        ,   141.33333333, 33313.        ],
       [33706.        , 33313.        ,   911.33333333]])

In [244]:
p**3-q**2

np.float64(-4.025379237414564e+27)

In [248]:
%timeit lib.linalg.eig(ST_test)

9.81 μs ± 12.1 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [260]:
%timeit eig_special_3d(test)

49.9 μs ± 58.8 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [250]:
%timeit lib.linalg.svd(ST_test)

8.38 μs ± 13.1 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [252]:
lib.linalg.svd(ST_test)[1]

array([97698.13160219,  2544.47932667,  1356.65227552])

In [254]:
lib.linalg.eig(ST_test)[0]

array([97698.13160219+0.j, -2544.47932667+0.j, -1356.65227552+0.j])

In [261]:
eig_special_3d(test)

array([-2544.7324, -1356.4141, 97698.125 ], dtype=float32)

In [256]:
test = np.float32(test)

In [300]:
input_shape = S.shape
S = S.reshape(6, -1)

In [384]:
S = S.compute()

In [298]:
S = np.float32(S)

In [301]:
S.shape

(6, 96000000)

In [302]:
S0 = S[0]
S1 = S[1]
S2 = S[2]
S3 = S[3]
S4 = S[4]
S5 = S[5]

S_full = lib.array([[S0,S3,S4],[S3,S1,S5],[S4,S5,S2]])

In [303]:
S_full.shape

(3, 3, 96000000)

In [285]:
t0 = time.time()
val = lib.stack([ lib.linalg.svd(S_full[:,:,i])[1] for i in range(S_full.shape[-1])])
npsvdtime = time.time() - t0
print(npsvdtime)

KeyboardInterrupt: 

In [370]:
t0 = time.time()
val = eig_special_3d(S)
QIM_time = time.time() - t0
print(QIM_time)

2.7653326988220215


In [390]:
t0 = time.time()
val_rob = eig_special_3d_robert(S)
val_rob = val_rob.compute()
rob_time = time.time() - t0
print(rob_time)

/gpfs/gb/data/id19/inhouse/robert/miniforge3/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in sqrt
  return self.func(*new_argspec)


8.791624546051025


In [387]:
import dask.array as lib

In [380]:
import numpy as np

In [288]:
S0

array([25224., 26477., 27776., ..., 29141., 29819., 30405.],
      shape=(96000000,), dtype=float32)

In [305]:
lib.trace(S_full[:,:,100])/3

np.float32(24310.334)

In [307]:
m = (S0[100]+S1[100]+S2[100])/3

In [291]:
S_full.shape

(3, 3, 96000000)

In [294]:
S0.shape

(96000000,)

In [308]:
m

np.float32(24310.334)

In [309]:
Sq = lib.subtract(S[:3], m)

In [319]:
Sq.shape

(3, 96000000)

In [322]:
s = lib.einsum("ij,ij->j", S[3:], S[3:])
s *= 2

In [324]:
p = lib.einsum("ij,ij->j", Sq, Sq)
p += s

In [328]:
p /= 6
# p = lib.sqrt(p)

In [313]:
AmI = S_full - m*lib.eye(3)[:,:,None]

In [318]:
AmI.shape

(3, 3, 96000000)

In [320]:
p_full = lib.sum(AmI**2, axis=(0,1))/6

In [321]:
p_full.shape

(96000000,)

In [338]:
p

array([1.2818574e+09, 1.2291273e+09, 1.1824777e+09, ..., 1.1938883e+09,
       1.2160846e+09, 1.2382317e+09], shape=(96000000,), dtype=float32)

In [339]:
p_full

array([1.28185742e+09, 1.22912740e+09, 1.18247767e+09, ...,
       1.19388829e+09, 1.21608470e+09, 1.23823167e+09], shape=(96000000,))

In [333]:
lib.linalg.det(AmI[:,:,0])

np.float64(57579032221568.67)

In [335]:
S0*S1

array([9.6131187e+08, 9.6662234e+08, 9.7363213e+08, ..., 8.5135430e+08,
       8.4167110e+08, 8.2434035e+08], shape=(96000000,), dtype=float32)

In [337]:
Sq.shape

(3, 96000000)

In [340]:
    S0q = Sq[0]
    S1q = Sq[1]
    S2q = Sq[2]
    
    det = S0q*S1q*S3 + S3*S4*S5 + S5*S3*S5 - S4*S3*S5 - S3*S3*S2q - S0q*S5*S5
    q = det/2

In [344]:
q**2

array([2.03237427e+26, 1.88801651e+26, 1.69642862e+26, ...,
       1.01505930e+26, 1.09078595e+26, 1.18579785e+26],
      shape=(96000000,), dtype=float32)

In [342]:
q.shape

(96000000,)

In [345]:
p**3

array([2.1062948e+27, 1.8569089e+27, 1.6534035e+27, ..., 1.7017315e+27,
       1.7984210e+27, 1.8984787e+27], shape=(96000000,), dtype=float32)

In [347]:
phi = lib.atan(lib.sqrt(p**3-q**2))/q/3

In [348]:
phi.shape

(96000000,)

In [349]:
    psqrt = lib.sqrt(p)
    cos = lib.cos(phi)
    sin = lib.sin(phi)

In [350]:
    v1 = m + 2*psqrt*cos
    v2 = m - lib.sqrt(p)*(cos + lib.sqrt(3)*sin)
    v3 = m - lib.sqrt(p)*(cos - lib.sqrt(3)*sin)

In [364]:
    # 3m = trace(A)
    m = (S0+S1+S2)/3
    
    
    # 6p = "sum of squares of elements of (A-mI)
    Sq = lib.subtract(S[:3], m)
    s = lib.einsum("ij,ij->j", S[3:], S[3:])
    s *= 2
    p = lib.einsum("ij,ij->j", Sq, Sq)
    p += s
    p /= 6
    
    #2q = det(A - mI)
    S0q = Sq[0]
    S1q = Sq[1]
    S2q = Sq[2]
    
    det = S0q*S1q*S2q + S3*S5*S4 + S4*S3*S5 - S4*S1q*S4 - S3*S3*S2q - S0q*S5*S5
    q = det/2
    
        #phi
    phi = lib.atan(lib.sqrt(p**3-q**2)/q)/3
    
    #eigenvalues
    psqrt = lib.sqrt(p)
    cos = lib.cos(phi)
    sin = lib.sin(phi)
    
    v1 = m + 2*psqrt*cos
    v2 = m - lib.sqrt(p)*(cos + lib.sqrt(3)*sin)
    v3 = m - lib.sqrt(p)*(cos - lib.sqrt(3)*sin)

/tmp/ipykernel_2926771/3291626933.py:22: RuntimeWarning: invalid value encountered in sqrt
  phi = lib.atan(lib.sqrt(p**3-q**2)/q)/3


In [365]:
valp = lib.array([v1,v2,v3])

In [360]:
val

array([[-9.8831562e+03, -7.7687109e+03, -5.7449336e+03, ...,
        -7.7966250e+03, -8.2020801e+03, -8.7584277e+03],
       [ 2.6673809e+03,  1.6195332e+03,  6.9767578e+02, ...,
        -4.3568359e+01, -1.1467969e+03, -2.3289082e+03],
       [ 1.0211477e+05,  1.0067018e+05,  9.9370242e+04, ...,
         9.8737211e+04,  9.9085898e+04,  9.9318328e+04]],
      shape=(3, 96000000), dtype=float32)

In [368]:
valp.max(axis=0).shape

(96000000,)